# Test PaperQA with Three Locally-Hosted NIM APIs

Prerequisites:
- Launch NIMs -> `launch_NIMs.sh`
- Install PaperQA -> `install_PQA.sh`

***Use the Jupyter Kernel built in last step.***

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
for _name in ("LiteLLM", "litellm"):
    _log = logging.getLogger(_name)
    _log.setLevel(logging.INFO)
    if not _log.handlers:
        _h = logging.StreamHandler(sys.stdout)
        _h.setLevel(logging.INFO)
        _h.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
        _log.addHandler(_h)
    _log.propagate = False

In [ ]:
# # Test if paperqa_nemotron is available
import paperqa
print(f"PaperQA location: {paperqa.__file__}")

# Log embedding call
from paperqa.llms import LiteLLMEmbeddingModel
_embed_log = logging.getLogger("LiteLLM")
_orig_embed = LiteLLMEmbeddingModel.embed_documents
async def _logged_embed(self, *args, **kwargs):
    texts = args[0] if args else kwargs.get("texts", [])
    n = len(texts) if texts else 0
    kind = "query" if n == 1 else "embed_documents"
    msg = f"LiteLLM embedding() model= {getattr(self, 'name', '?')}; ({kind}) n={n}"
    _embed_log.info(msg)
    if not _embed_log.handlers:
        print(f"INFO: {msg}")
    return await _orig_embed(self, *args, **kwargs)
LiteLLMEmbeddingModel.embed_documents = _logged_embed

try:
    import paperqa_nemotron
    from paperqa_nemotron import parse_pdf_to_pages
    print(f"✅ paperqa_nemotron location: {paperqa_nemotron.__file__}")
    print(f"✅ parse_pdf_to_pages available: {parse_pdf_to_pages}")
except ImportError as e:
    print(f"❌ paperqa_nemotron NOT installed!")
    print(f"   Error: {e}")
    print(f"   Run: pip install -e '.[local,pymupdf,nemotron]'")

## Step 1: Load a PDF Paper

In [ ]:
# Create papers directory
!mkdir -p papers

**Upload a PDF to the `papers/` folder, then run the cell below to auto-detect it:**

I tested two PDFs: 
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [Acid-sensing ion channels 1a (ASIC1a) inhibit neuromuscular transmission in female mice](https://pubmed.ncbi.nlm.nih.gov/24336653/)

In [ ]:
import os
import glob

# Auto-detect PDF files in the papers folder
pdf_files = sorted(glob.glob("papers/*.pdf"))

if pdf_files:
    print(f"📁 Found {len(pdf_files)} PDF(s) in papers/:")
    for i, pdf in enumerate(pdf_files):
        size_mb = os.path.getsize(pdf) / (1024 * 1024)
        print(f"   [{i}] {pdf} ({size_mb:.2f} MB)")
    
    # Use the first PDF, or change the index to select a different one
    paper_path = pdf_files[0]  # <-- Change index to select different PDF
    print(f"\n✅ Selected: {paper_path}")
else:
    print("❌ No PDFs found in papers/ folder!")
    print("   Upload a PDF to papers/ and re-run this cell.")
    paper_path = None

## Step 2: Configure Locally-Hosted Endpoints

We configure three locally-hosted NIMs (started via `launch_NIMs.sh`):

- **Parse** nvidia/nemotron-parse -> PDF parser (localhost:8002)
- **Embedding** nvidia/llama-3.2-nv-embedqa-1b-v2 -> embedding (localhost:8003)
- **VLM** nvidia/nemotron-nano-12b-v2-vl -> enrichment_llm, summary_llm, llm, agent_llm (localhost:8004)
  
*Right-side terms match paper-qa `Settings` / `ParsingSettings` field names.*

In [ ]:
# =============================================================================
# NIM CONFIGURATION (Parse=8002, Embedding=8003, VLM=8004)
# =============================================================================
import os
SELFHOST_API_KEY = "dummy"

# ----- Parse -----
PARSE_API_BASE = "http://localhost:8002/v1"
PARSE_API_KEY = "dummy"
PARSE_MODEL_NAME = "nvidia/nemotron-parse"
PARSE_COMPLETION_KWARGS = {
    "temperature": 0,
    "max_tokens": 8995,
}

# ----- Embedding --------------------
EMBEDDING_API_BASE = "http://localhost:8003/v1"
EMBEDDING_MODEL = "nvidia/llama-3.2-nv-embedqa-1b-v2"

# ----- VLM --------------------
VLM_API_BASE = "http://localhost:8004/v1"
VLM_MODEL = "nvidia/nemotron-nano-12b-v2-vl"
CUSTOM_VLM_NAME = "selfhost-nemotron-vlm"

print("\n=== Locally-Hosted NIM Endpoints ===")
print(f"Nemotron-Parse: {PARSE_API_BASE} ({PARSE_MODEL_NAME})")
print(f"Embedding: {EMBEDDING_API_BASE}  ({EMBEDDING_MODEL})")
print(f"VLM: {VLM_API_BASE}  ({VLM_MODEL})")

## Step 3: Create PaperQA Settings

In [ ]:
from paperqa import Settings
from paperqa.settings import AgentSettings, IndexSettings, AnswerSettings, ParsingSettings
from paperqa_nemotron import parse_pdf_to_pages
import pathlib

# VLM config for enrichment_llm, summary_llm, llm, agent_llm (locally-hosted nemotron-nano-12b-v2-vl on 8004)
nvidia_vlm_config = {
    "model_list": [
        {
            "model_name": CUSTOM_VLM_NAME,
            "litellm_params": {
                "model": f"openai/{VLM_MODEL}",
                "api_base": VLM_API_BASE,
                "api_key": SELFHOST_API_KEY,
                "temperature": 0,
                "max_tokens": 2048,
            },
        }
    ]
}

# Embedding config (locally-hosted llama-3.2-nv-embedqa-1b-v2 on 8003)
nvidia_embedding_config = {
    "kwargs": {
        "api_base": EMBEDDING_API_BASE,
        "api_key": SELFHOST_API_KEY,
        "encoding_format": "float",
        "input_type": "passage",     
    }
}

# ParsingSettings with Nemotron-Parse NIM
parsing_settings = ParsingSettings(
    # Avoid Semantic Scholar / metadata API rate limits (429)
    use_doc_details=False,    
    # Use nemotron-parse as the PDF parser
    parse_pdf=parse_pdf_to_pages,
    
    # reader_config is passed to parse_pdf_to_pages(); api_params go to LiteLLM
    reader_config={
        "chunk_chars": 5000,
        "overlap": 250,
        "dpi": 150,  # Image resolution for PDF rendering
        
        "api_params": {
            "api_base": PARSE_API_BASE,       
            "api_key": PARSE_API_KEY,         
            "model_name": PARSE_MODEL_NAME, 
            **PARSE_COMPLETION_KWARGS, 
        }
    },
    
    # Enrichment LLM for multimodal (self-hosted VLM on 8004)
    enrichment_llm=CUSTOM_VLM_NAME,
    enrichment_llm_config=nvidia_vlm_config,
    multimodal=True,  # Enable multimodal to use nemotron-parse's full capabilities
)

# Full settings
settings = Settings(
    # LLMs
    llm=CUSTOM_VLM_NAME,
    llm_config=nvidia_vlm_config,
    summary_llm=CUSTOM_VLM_NAME,
    summary_llm_config=nvidia_vlm_config,
    
    # Embedding (self-hosted on 8003)
    embedding=f"openai/{EMBEDDING_MODEL}",
    embedding_config=nvidia_embedding_config,
    
    # Temperature for all LLMs (answer, summary, agent, enrichment) unless overridden in their config
    temperature=0,
    # Global logging level 0-3 for LLM/embedding calls (not per-model)
    verbosity=3,
    
    answer=AnswerSettings(
        evidence_k=5,
        answer_max_sources=3,
    ),
    
    # Use our nemotron-parse settings
    parsing=parsing_settings,
    
    agent=AgentSettings(
        agent_llm=CUSTOM_VLM_NAME,
        agent_llm_config=nvidia_vlm_config,
        index=IndexSettings(
            paper_directory=pathlib.Path.cwd() / "papers",
        ),
    ),
)

print("✅ PaperQA Settings created!")
print(f"   PDF Parser: {parsing_settings.parse_pdf}")
print(f"   Nemotron Parse API Base: {parsing_settings.reader_config['api_params']['api_base']}")
print(f"   Multimodal: {parsing_settings.multimodal}")
print(f"   Embedding: {settings.embedding}")
print(f"   LLM: {settings.llm}")

## Step 4: Add Paper with Nemotron-Parse

In [ ]:
from paperqa import Docs

# Create a Docs object and add the paper
docs = Docs()

print("Adding paper to Docs (using nemotron-parse)...")
# print("This will call your Docker NIM on localhost:8002\n")

try:
    await docs.aadd(paper_path, settings=settings)
    print(f"\n✅ Paper added successfully!")
    print(f"   Total docs: {len(docs.docs)}")
    
    # Show what was added
    for doc_key, doc in docs.docs.items():
        print(f"   - {doc.docname}: {doc.citation[:80]}...")
        
except Exception as e:
    print(f"❌ Failed to add paper!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Step 5: `docs.aquery()` -> non-agent, no tools

In [ ]:
# Query the docs
print("Querying docs...")

try:
    session = await docs.aquery(
        "What experiments are carried out?",
        settings=settings,
    )
    
    print("✅ Query SUCCESS!")
    print("\n" + "=" * 60)
    print("Question:")
    print("=" * 60)
    print(session.question)
    
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(session.answer)
    
    print("\n" + "=" * 60)
    print("References:")
    print("=" * 60)
    print(session.references)
    
except Exception as e:
    print(f"❌ Query FAILED!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Step 6 `agent_query()` -> agent, tools


In [ ]:
from paperqa.agents.main import agent_query

agent_question = "What experiments are carried out?"
print(f"Running agent_query with docs (pre-loaded paper)...")
print(f"Question: {agent_question}\n")

try:
    response = await agent_query(agent_question, settings, docs=docs)
    print("✅ agent_query SUCCESS!")
    print(f"Status: {response.status}")
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(response.session.answer or "(no answer)")
except Exception as e:
    print(f"❌ agent_query failed: {e}")
    import traceback
    traceback.print_exc()

## Step 6.5: `ask()` -> agent, tools

Test the full agent workflow:

In [ ]:
from paperqa import ask

print("Running agent-based query with ask()...")
print("This will search, gather evidence, and generate an answer.\n")

try:
    response = await ask(
        "What are novel findings?",
        settings=settings,
    )
    
    print("✅ Agent query SUCCESS!")
    print(f"\n--- Status: {response.status} ---")
    
    print("\n" + "=" * 60)
    print("Question:")
    print("=" * 60)
    print(response.session.question)
    
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(response.session.answer)
    
    print("\n" + "=" * 60)
    print("Contexts Used (top 3):")
    print("=" * 60)
    for i, ctx in enumerate(response.session.contexts[:3], 1):
        print(f"\nContext {i} (score: {ctx.score}):")
        print(f"  Source: {ctx.text.name}")
        print(f"  Summary: {ctx.context[:200]}...")
        
except Exception as e:
    print(f"❌ Agent query FAILED!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## (Optional) Step 7: Verify Nemotron-Parse Was Used

Create a fresh Docs object and inspect parsing metadata to confirm nemotron-parse was used:

In [ ]:
from paperqa import Docs
from paperqa.readers import read_doc

print("=" * 70)
print("VERIFYING NEMOTRON-PARSE WAS USED")
print("=" * 70)

# Create a fresh Docs object
verify_docs = Docs()

print(f"\n📄 Parsing: {paper_path}")
print("   (Watch your Docker NIM logs for POST requests)\n")

try:
    # Add the paper - this will call nemotron-parse
    await verify_docs.aadd(paper_path, settings=settings)
    
    print("✅ Paper added successfully!\n")
    
    # Inspect the parsed content
    print("=" * 70)
    print("PARSING METADATA EVIDENCE")
    print("=" * 70)
    
    # Check texts for media (images/tables detected by nemotron-parse)
    total_media = 0
    media_types = {}
    enriched_count = 0
    
    for text in verify_docs.texts:
        for media in text.media:
            total_media += 1
            media_type = media.info.get('type', 'unknown')
            media_types[media_type] = media_types.get(media_type, 0) + 1
            if media.info.get('enriched_description'):
                enriched_count += 1
    
    print(f"\n📊 Media Detection Summary:")
    print(f"   Total media items detected: {total_media}")
    print(f"   Media items with enrichment: {enriched_count}")
    
    if media_types:
        print(f"\n📋 Media Types (from nemotron-parse classification):")
        for mtype, count in sorted(media_types.items()):
            print(f"   • {mtype}: {count}")
        print("\n   ⬆️ These type classifications (Picture, Table, etc.) are from")
        print("      nemotron-parse's markdown_bbox tool - NOT from PyMuPDF!")
    else:
        print("\n   ⚠️ No media detected (document may be text-only)")
    
    # Show sample enriched descriptions WITH IMAGES (side-by-side)
    print("\n" + "=" * 70)
    print("MEDIA GALLERY: CROPPED IMAGES WITH ENRICHED DESCRIPTIONS")
    print("=" * 70)
    
    from PIL import Image
    from IPython.display import display, HTML
    import io
    import base64
    
    # Collect all media items, prioritizing those with enriched descriptions
    all_media = []
    for text in verify_docs.texts:
        for media in text.media:
            all_media.append(media)
    
    # Sort: enriched items first, then by page number
    all_media.sort(key=lambda m: (
        0 if m.info.get('enriched_description') else 1,
        m.info.get('page_num', 999)
    ))
    
    # Ensure we show at least 3 items (or all if less than 3)
    min_display = 6
    items_to_show = min(max(min_display, 3), len(all_media), 7)  # Show 3-6 items
    
    print(f"\n📸 Showing {items_to_show} of {len(all_media)} detected media items:\n")
    
    if all_media:
        for idx, media in enumerate(all_media[:items_to_show]):
            # Convert image to base64 for HTML embedding
            try:
                img = Image.open(io.BytesIO(media.data))
                
                # Resize if needed (max width 400px for side-by-side display)
                max_width = 400
                if img.width > max_width:
                    ratio = max_width / img.width
                    new_size = (int(img.width * ratio), int(img.height * ratio))
                    img = img.resize(new_size, Image.Resampling.LANCZOS)
                
                # Convert to base64
                buffered = io.BytesIO()
                img.save(buffered, format="PNG")
                img_base64 = base64.b64encode(buffered.getvalue()).decode()
                
                media_type = media.info.get('type', 'Unknown')
                page_num = media.info.get('page_num', '?')
                enriched_desc = media.info.get('enriched_description', '')
                
                # Create HTML for side-by-side display
                html_content = f"""
                <div style="display: flex; align-items: flex-start; margin: 20px 0; padding: 15px; 
                            border: 2px solid #4a90d9; border-radius: 10px; background: #f8f9fa;">
                    <div style="flex-shrink: 0; margin-right: 20px;">
                        <div style="background: #4a90d9; color: white; padding: 5px 10px; 
                                    border-radius: 5px 5px 0 0; font-weight: bold; text-align: center;">
                            #{idx + 1} {media_type} (Page {page_num})
                        </div>
                        <img src="data:image/png;base64,{img_base64}" 
                             style="border: 1px solid #ddd; border-radius: 0 0 5px 5px; max-width: 400px;"
                             alt="{media_type}"/>
                        <div style="font-size: 11px; color: #666; margin-top: 5px;">
                            Size: {img.width}×{img.height}px
                        </div>
                    </div>
                    <div style="flex-grow: 1;">
                        <div style="font-weight: bold; color: #333; margin-bottom: 10px; font-size: 14px;">
                            📝 Enriched Description:
                        </div>
                        <div style="background: white; padding: 12px; border-radius: 8px; 
                                    border: 1px solid #e0e0e0; line-height: 1.6; font-size: 13px;">
                            {enriched_desc if enriched_desc else '<em style="color: #999;">No enrichment available</em>'}
                        </div>
                    </div>
                </div>
                """
                display(HTML(html_content))
                
            except Exception as img_err:
                print(f"   ⚠️ Could not display media #{idx + 1}: {img_err}")
    else:
        print("\n   No media items found in this document.")
    
    # Show a sample text chunk
    print("\n" + "=" * 70)
    print("SAMPLE PARSED TEXT (first chunk)")
    print("=" * 70)
    if verify_docs.texts:
        sample_text = verify_docs.texts[0].text[:500]
        print(f"\n{sample_text}...")
    
    print("\n" + "=" * 70)
    print("CONCLUSION")
    print("=" * 70)
    if total_media > 0 and media_types:
        print("\n✅ CONFIRMED: Nemotron-parse was used!")
        print("   Evidence: Media items with type classifications detected.")
        print("   (PyMuPDF doesn't provide these structured type labels)")
    else:
        print("\n⚠️ Could not confirm nemotron-parse from media detection.")
        print("   Check your Docker NIM logs for POST requests as direct evidence.")

except Exception as e:
    print(f"❌ Verification failed: {e}")
    import traceback
    traceback.print_exc()